# GHS Hazard Classification — full-dataset run on Google Colab

**Project:** Interpretable Machine Learning for Predicting GHS Chemical Hazard
Classifications
**Researcher:** Sareer Ahmad, MSc Physical Chemistry, University of Peshawar

---

## What this notebook does

The local run modelled **40,000 compounds** because that machine had 7.9 GB of
RAM. This notebook trains on the **full 243,323-compound cleaned dataset**.

## Why XGBoost only

XGBoost was already selected as the best model locally (mean AUC 0.890, mean
MCC 0.517). It is also the only one of the three algorithms that *can* scale:

| Model | Memory at full dataset | Feasible on Colab? |
|---|---|---|
| **XGBoost** (`hist`) | **~0.8 GB** | **yes, even free tier** |
| Random Forest | ~28 GB | Pro+ only |
| SVM (RBF kernel) | **304 GB** | no — at any tier |

The SVM figure is not a hardware problem. An RBF kernel matrix is n x n, so at
195,000 training compounds it is 304 GB regardless of the machine. That is a
property of the algorithm.

## Before you start

1. **Runtime → Change runtime type → CPU** is fine. A GPU speeds XGBoost up but
   is not required; set `USE_GPU = True` below if you have one.
2. Free-tier sessions disconnect after ~90 minutes idle and 12 hours total.
   Results are written to your Google Drive as they are produced, so a
   disconnect does not lose completed work.
3. You need **one file** from the local project:
   `STEP3_cleaned_ghs_dataset.csv` (~78 MB). Section 3 will help you
   load it, either from Google Drive or by uploading it directly.


---
## 1. Install the packages

Colab already has pandas, numpy, scikit-learn and matplotlib. Only RDKit,
XGBoost and SHAP need adding. Takes about a minute.

In [ ]:
!pip install -q rdkit xgboost shap
import rdkit, xgboost, shap, sklearn, numpy, pandas
print("rdkit       ", rdkit.__version__)
print("xgboost     ", xgboost.__version__)
print("shap        ", shap.__version__)
print("scikit-learn", sklearn.__version__)

---
## 2. Check what hardware Colab gave you

Free tier is about 12.7 GB of RAM; the high-RAM Pro runtime is about 25 GB.
XGBoost needs far less than either, so this is a sanity check rather than a
gate.

In [ ]:
import os, psutil, multiprocessing
ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM        : {ram_gb:.1f} GB")
print(f"CPU cores  : {multiprocessing.cpu_count()}")
print(f"Disk free  : {psutil.disk_usage('/').free / 1e9:.1f} GB")

try:
    import subprocess
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU        :", out.stdout.strip() or "none")
except Exception:
    print("GPU        : none")

# XGBoost needs roughly the feature matrix plus its histograms.
print(f"\nEstimated need for 243,323 x 817 float32: "
      f"{243323 * 817 * 4 / 1e9:.2f} GB for the matrix")
print("Comfortable on any Colab tier." if ram_gb > 8 else "Tight - use a high-RAM runtime.")

---
## 3. Load the cleaned dataset

You need **one file** from the local project:
`D:\GHS_Project\STEP3_cleaned_ghs_dataset.csv` (78 MB).

This is the *cleaned* dataset, so the PubChem harvest, SMILES validation,
InChIKey deduplication and multi-source majority voting from Steps 2 and 3 are
already done. Nothing is re-downloaded.

**The cell below handles both ways of getting the file in — just run it.**

| | Google Drive | Direct upload |
|---|---|---|
| Setup | Put the CSV in Drive first | Nothing to do beforehand |
| After a disconnect | File is still there | Re-upload the 78 MB |
| Results | Saved to Drive permanently | Lost when the session ends |

**Drive is recommended** — a free Colab session disconnects after about 90
minutes idle, and without Drive you lose both the file and your results.

*To use Drive:* open [drive.google.com](https://drive.google.com), make a
folder called `GHS_Project`, and drag the CSV into it. Then run the cell and
click through the permission prompt.

*To use direct upload:* run the cell, decline the Drive prompt if it appears,
and a file-picker will open instead.


In [ ]:
import os, pandas as pd

DRIVE_CSV = '/content/drive/MyDrive/GHS_Project/STEP3_cleaned_ghs_dataset.csv'
data_path, OUT_DIR = None, '/content/ghs_outputs'

# ---- try Google Drive first -------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_CSV):
        data_path = DRIVE_CSV
        OUT_DIR = '/content/drive/MyDrive/GHS_Project/colab_full_run'
        print(f"Found the dataset in Drive: {DRIVE_CSV}")
    else:
        print(f"Drive mounted, but no file at:\n  {DRIVE_CSV}")
        print("Falling back to direct upload.")
except Exception as e:
    print(f"Drive not available ({type(e).__name__}). Falling back to direct upload.")

# ---- otherwise ask the browser for the file --------------------------------
if data_path is None:
    local = 'STEP3_cleaned_ghs_dataset.csv'
    if os.path.exists(local):
        data_path = local
        print(f"Using the copy already in this session: {local}")
    else:
        print("\nSelect STEP3_cleaned_ghs_dataset.csv from your computer.")
        print("It is 78 MB, so the upload takes a few minutes.")
        from google.colab import files
        uploaded = files.upload()
        data_path = list(uploaded.keys())[0]
    print("\nNOTE: results will be written to this temporary session only and")
    print("will be LOST when it disconnects. Download anything you need, or")
    print("put the CSV in Drive and re-run this cell.")

os.makedirs(OUT_DIR, exist_ok=True)
df = pd.read_csv(data_path, low_memory=False)
print(f"\nLoaded {len(df):,} compounds x {len(df.columns)} columns")
print(f"Outputs -> {OUT_DIR}")

if len(df) < 200000:
    print(f"\nWARNING: expected about 243,000 compounds but got {len(df):,}.")
    print("Check you uploaded STEP3_cleaned_ghs_dataset.csv and not")
    print("STEP3_modelling_subset.csv (which is the 40,000-compound subset).")

---
## 4. Project configuration

The nine label columns follow the official United Nations pictogram numbering.
The seed is 42 everywhere, exactly as in the local run, so results are
comparable.

In [ ]:
import random, numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

USE_GPU = False   # set True if section 2 reported a GPU

GHS_LABEL_COLUMNS = [
    "GHS01_Explosive", "GHS02_Flammable", "GHS03_Oxidising",
    "GHS04_CompressedGas", "GHS05_Corrosive", "GHS06_AcuteToxicity",
    "GHS07_Irritant", "GHS08_HealthHazard", "GHS09_Environmental",
]
GHS_TRUE_MEANING = {
    "GHS01_Explosive": "Explosive", "GHS02_Flammable": "Flammable",
    "GHS03_Oxidising": "Oxidiser", "GHS04_CompressedGas": "Compressed gas",
    "GHS05_Corrosive": "Corrosive", "GHS06_AcuteToxicity": "Acute toxicity",
    "GHS07_Irritant": "Irritant / harmful",
    "GHS08_HealthHazard": "Serious health hazard",
    "GHS09_Environmental": "Environmental hazard",
}

missing = [c for c in GHS_LABEL_COLUMNS if c not in df.columns]
if missing:
    raise SystemExit(
        f"These label columns are missing: {missing}\n"
        f"If your copy predates the column rename, run "
        f"src/migrate_column_names.py locally before uploading.")

print(df[GHS_LABEL_COLUMNS].sum().to_string())
print(f"\nTotal compounds: {len(df):,}")

---
## 5. Compute descriptors for all 243,323 compounds

The same 1218 descriptors as the local run: 19 physicochemical, 1024 Morgan
(ECFP4), 167 MACCS keys and 8 topological.

**This is the slow step — roughly 25–40 minutes** depending on how many cores
Colab gives you. The result is cached to Drive, so if the session drops you can
re-run this cell and it will reload rather than recompute.

In [ ]:
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, GraphDescriptors
RDLogger.DisableLog("rdApp.*")

PHYSCHEM = ["MolWt","ExactMolWt","MolLogP","TPSA","NumHDonors","NumHAcceptors",
            "NumRotatableBonds","NumAromaticRings","NumSaturatedRings",
            "NumAliphaticRings","RingCount","FractionCSP3","HeavyAtomCount",
            "NumHeteroatoms","NOCount","NHOHCount","LabuteASA","BalabanJ","BertzCT"]
TOPO = ["Chi0","Chi1","Chi2n","Chi3n","Chi4n","Kappa1","Kappa2","Kappa3"]
MORGAN_BITS, MACCS_BITS = 1024, 167
N_FEATURES = len(PHYSCHEM) + MORGAN_BITS + MACCS_BITS + len(TOPO)

FEATURE_NAMES = (PHYSCHEM
                 + [f"Morgan_{i}" for i in range(MORGAN_BITS)]
                 + [f"MACCS_{i}" for i in range(MACCS_BITS)]
                 + TOPO)

# Functions are resolved by NAME, not stored as objects: RDKit's compiled
# descriptor functions cannot be pickled and so cannot be sent to workers.
_PHYS_FN = [getattr(Descriptors, n) for n in PHYSCHEM]
_TOPO_FN = [getattr(GraphDescriptors, n) for n in TOPO]

def descriptors_for(smiles):
    """Turn one SMILES string into one row of the feature matrix."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    row = np.full(N_FEATURES, np.nan, dtype=np.float32)
    p = 0
    for fn in _PHYS_FN:
        try:
            v = fn(mol); row[p] = v if np.isfinite(v) else np.nan
        except Exception:
            pass
        p += 1
    try:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=MORGAN_BITS)
        row[p:p+MORGAN_BITS] = np.frombuffer(fp.ToBitString().encode(),
                                             dtype=np.uint8) - 48
    except Exception:
        row[p:p+MORGAN_BITS] = 0
    p += MORGAN_BITS
    try:
        mk = MACCSkeys.GenMACCSKeys(mol)
        row[p:p+MACCS_BITS] = np.frombuffer(mk.ToBitString().encode(),
                                            dtype=np.uint8) - 48
    except Exception:
        row[p:p+MACCS_BITS] = 0
    p += MACCS_BITS
    for fn in _TOPO_FN:
        try:
            v = fn(mol); row[p] = v if np.isfinite(v) else np.nan
        except Exception:
            pass
        p += 1
    return row


CACHE_X = os.path.join(OUT_DIR, "colab_X_full.npy")
CACHE_KEEP = os.path.join(OUT_DIR, "colab_keep_full.npy")

if os.path.exists(CACHE_X):
    X_raw = np.load(CACHE_X)
    keep = np.load(CACHE_KEEP)
    print(f"Reloaded cached descriptors: {X_raw.shape}")
else:
    from joblib import Parallel, delayed
    from tqdm.auto import tqdm

    smiles_col = ("CanonicalSMILES_RDKit"
                  if "CanonicalSMILES_RDKit" in df.columns else "SMILES")
    smiles = df[smiles_col].tolist()

    def chunk_job(block):
        return [descriptors_for(s) for s in block]

    CHUNK = 500
    blocks = [smiles[i:i+CHUNK] for i in range(0, len(smiles), CHUNK)]
    n_jobs = max(1, multiprocessing.cpu_count() - 1)
    print(f"Computing {N_FEATURES} descriptors for {len(smiles):,} molecules "
          f"on {n_jobs} workers...")

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(chunk_job)(b) for b in tqdm(blocks))

    X_raw = np.zeros((len(smiles), N_FEATURES), dtype=np.float32)
    keep = np.ones(len(smiles), dtype=bool)
    pos = 0
    for block in results:
        for row in block:
            if row is None:
                keep[pos] = False
            else:
                X_raw[pos] = row
            pos += 1

    np.save(CACHE_X, X_raw)
    np.save(CACHE_KEEP, keep)
    print(f"Done. {int((~keep).sum())} molecules failed and were dropped.")

X_raw = X_raw[keep]
df = df[keep].reset_index(drop=True)
print(f"Feature matrix: {X_raw.shape}  ({X_raw.nbytes/1e9:.2f} GB)")

---
## 6. Impute and variance-filter

Median imputation first, then the variance filter — a column containing a NaN
has undefined variance, so filtering first would silently discard usable
descriptors.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

n_nan = int(np.isnan(X_raw).sum())
if n_nan:
    X_raw = SimpleImputer(strategy="median",
                          keep_empty_features=True).fit_transform(X_raw).astype(np.float32)
X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)

selector = VarianceThreshold(threshold=0.01)
X = selector.fit_transform(X_raw).astype(np.float32)
kept_names = [n for n, k in zip(FEATURE_NAMES, selector.get_support()) if k]
del X_raw

y = df[GHS_LABEL_COLUMNS].to_numpy().astype(np.int8)
print(f"{n_nan:,} missing values imputed")
print(f"Feature matrix after variance filter: {X.shape}")
print(f"Label matrix: {y.shape}")

---
## 7. Bemis-Murcko scaffold split

Identical logic to the local run, including the two details that matter:

- **Acyclic compounds each form their own group.** Pooling every ring-free
  molecule into one group would put most industrial solvents in a single split.
- **Largest groups are assigned first.** Some scaffolds (plain benzene) are
  shared by thousands of compounds; meeting one after the training quota is
  full would push it into the test set and wreck the 80/10/10 proportions.

Takes about 10–15 minutes on the full dataset.

In [ ]:
import collections
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm.auto import tqdm

smiles_col = ("CanonicalSMILES_RDKit"
              if "CanonicalSMILES_RDKit" in df.columns else "SMILES")

scaffolds, n_acyclic = [], 0
for s in tqdm(df[smiles_col], desc="scaffolds"):
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        scaffolds.append(f"__unparsed_{len(scaffolds)}__"); continue
    try:
        core = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    except Exception:
        core = ""
    if not core:                       # no rings
        core = Chem.MolToSmiles(mol)   # its own group
        n_acyclic += 1
    scaffolds.append(core)

groups = collections.defaultdict(list)
for i, sc in enumerate(scaffolds):
    groups[sc].append(i)

group_list = list(groups.values())
rng = np.random.RandomState(RANDOM_SEED)
rng.shuffle(group_list)
group_list.sort(key=len, reverse=True)      # largest first

n_total = len(scaffolds)
n_train_target, n_val_target = int(0.8 * n_total), int(0.1 * n_total)
train_idx, val_idx, test_idx = [], [], []
for g in group_list:
    if len(train_idx) + len(g) <= n_train_target:   train_idx.extend(g)
    elif len(val_idx) + len(g) <= n_val_target:     val_idx.extend(g)
    else:                                           test_idx.extend(g)

train_idx = np.sort(np.array(train_idx))
val_idx   = np.sort(np.array(val_idx))
test_idx  = np.sort(np.array(test_idx))

sc = np.asarray(scaffolds, dtype=object)
overlap = ((set(sc[train_idx]) & set(sc[test_idx]))
           | (set(sc[train_idx]) & set(sc[val_idx]))
           | (set(sc[val_idx]) & set(sc[test_idx])))
print(f"distinct scaffolds : {len(groups):,}  ({n_acyclic:,} acyclic)")
print(f"train {len(train_idx):,} ({100*len(train_idx)/n_total:.1f}%) | "
      f"val {len(val_idx):,} ({100*len(val_idx)/n_total:.1f}%) | "
      f"test {len(test_idx):,} ({100*len(test_idx)/n_total:.1f}%)")
print(f"scaffolds shared between splits: {len(overlap)}  "
      f"{'PASS' if not overlap else 'FAIL - investigate'}")

np.save(os.path.join(OUT_DIR, "colab_train_idx.npy"), train_idx)
np.save(os.path.join(OUT_DIR, "colab_val_idx.npy"), val_idx)
np.save(os.path.join(OUT_DIR, "colab_test_idx.npy"), test_idx)

---
## 8. Train XGBoost on the full dataset

One model per hazard, with `scale_pos_weight` set to the negative-to-positive
ratio. `tree_method="hist"` keeps memory low — this is what makes the full
dataset tractable where a Random Forest would need ~28 GB.

Roughly 20–40 minutes on CPU, considerably less on a GPU.

In [ ]:
import time, joblib
from xgboost import XGBClassifier

X_train, y_train = X[train_idx], y[train_idx]
print(f"Training on {X_train.shape[0]:,} compounds x {X_train.shape[1]:,} features")
print(f"Training matrix: {X_train.nbytes/1e9:.2f} GB\n")

models, started = [], time.time()
for i, col in enumerate(GHS_LABEL_COLUMNS):
    yc = y_train[:, i].astype(int)
    n_pos, n_neg = int(yc.sum()), int(len(yc) - yc.sum())
    if n_pos < 2:
        print(f"{col:<22} skipped - only {n_pos} positives")
        models.append(None); continue

    t0 = time.time()
    m = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg / n_pos,
        random_state=RANDOM_SEED, tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        eval_metric="aucpr", verbosity=0, n_jobs=-1)
    m.fit(X_train, yc)
    models.append(m)
    print(f"{col:<22} n+={n_pos:>7,}  spw={n_neg/n_pos:>8.1f}  "
          f"{(time.time()-t0)/60:>5.1f} min")

print(f"\nTotal training time: {(time.time()-started)/60:.1f} minutes")
joblib.dump(models, os.path.join(OUT_DIR, "colab_xgb_full.pkl"), compress=3)

---
## 9. Evaluate, with thresholds calibrated on validation only

Thresholds are fitted on the validation split and then applied unchanged to the
test split. Fitting them on the test set would inflate every score.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             precision_recall_curve, confusion_matrix)

def positive_proba(models, Xm):
    """Probability of the positive class for all nine hazards."""
    out = []
    for m in models:
        out.append(np.zeros(Xm.shape[0]) if m is None
                   else m.predict_proba(Xm)[:, 1])
    return np.column_stack(out)

P_val  = positive_proba(models, X[val_idx])
P_test = positive_proba(models, X[test_idx])
y_val, y_test = y[val_idx], y[test_idx]

rows = []
for i, col in enumerate(GHS_LABEL_COLUMNS):
    yv, yt = y_val[:, i], y_test[:, i]
    if len(np.unique(yt)) < 2:
        continue

    # threshold that maximises F1 on the validation split
    pr, rc, th = precision_recall_curve(yv, P_val[:, i])
    pr, rc = pr[:-1], rc[:-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        f1s = np.nan_to_num(2 * pr * rc / (pr + rc))
    thr = float(th[int(np.argmax(f1s))]) if len(th) else 0.5

    pred = (P_test[:, i] >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    rows.append({
        "GHS_Column": col, "Meaning": GHS_TRUE_MEANING[col],
        "N_Test_Positive": int(yt.sum()),
        "AUC_ROC": round(roc_auc_score(yt, P_test[:, i]), 4),
        "Average_Precision": round(average_precision_score(yt, P_test[:, i]), 4),
        "F1": round(f1_score(yt, pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(yt, pred), 4),
        "Precision": round(precision_score(yt, pred, zero_division=0), 4),
        "Recall": round(recall_score(yt, pred, zero_division=0), 4),
        "Specificity": round(tn / (tn + fp) if (tn + fp) else 0.0, 4),
        "Threshold": round(thr, 4),
    })

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_DIR, "colab_full_results.csv"), index=False)

print("FULL-DATASET RESULTS (XGBoost, scaffold-split test set)")
print("=" * 96)
print(results[["GHS_Column","N_Test_Positive","AUC_ROC","F1","MCC",
               "Precision","Recall"]].to_string(index=False))
print("=" * 96)
print(f"Mean AUC-ROC: {results['AUC_ROC'].mean():.4f}")
print(f"Mean MCC    : {results['MCC'].mean():.4f}")

---
## 10. Compare against the 40,000-compound local run

This is the comparison that answers the reviewer's question: did training on
six times more data actually change anything?

The local bootstrap 95% confidence interval had a half-width of **0.0139**, so
any per-class difference smaller than that is not meaningful.

In [ ]:
LOCAL_40K = {   # XGBoost, 40,000-compound modelling subset
    "GHS01_Explosive": 0.9733, "GHS02_Flammable": 0.9558,
    "GHS03_Oxidising": 0.9966, "GHS04_CompressedGas": 0.9981,
    "GHS05_Corrosive": 0.8680, "GHS06_AcuteToxicity": 0.7827,
    "GHS07_Irritant": 0.7672,  "GHS08_HealthHazard": 0.8382,
    "GHS09_Environmental": 0.8309,
}
CI_HALF_WIDTH = 0.0139

comp = results[["GHS_Column", "AUC_ROC"]].copy()
comp["AUC_local_40k"] = comp["GHS_Column"].map(LOCAL_40K)
comp["Difference"] = (comp["AUC_ROC"] - comp["AUC_local_40k"]).round(4)
comp["Meaningful"] = np.where(comp["Difference"].abs() > CI_HALF_WIDTH,
                              "yes", "within noise")
comp = comp.rename(columns={"AUC_ROC": "AUC_full_243k"})
comp.to_csv(os.path.join(OUT_DIR, "colab_vs_local_comparison.csv"), index=False)

print(comp.to_string(index=False))
print(f"\nMean AUC  full 243k : {comp['AUC_full_243k'].mean():.4f}")
print(f"Mean AUC  local 40k : {comp['AUC_local_40k'].mean():.4f}")
print(f"Change              : {comp['AUC_full_243k'].mean() - comp['AUC_local_40k'].mean():+.4f}")

n_meaningful = int((comp["Meaningful"] == "yes").sum())
print(f"\nClasses changed by more than the bootstrap CI: {n_meaningful} of 9")
if n_meaningful == 0:
    print("\nCONCLUSION: training on the full dataset produced no change larger")
    print("than the uncertainty of the estimate. The 40,000-compound subset was")
    print("sufficient, and the memory constraint did not limit the published")
    print("results. Report this as a demonstrated sufficiency, not a limitation.")
else:
    print("\nCONCLUSION: some classes genuinely improved. Report the")
    print("full-dataset numbers as the headline result and keep the 40,000")
    print("comparison as evidence of the effect of dataset size.")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(comp)); w = 0.38
ax.bar(x - w/2, comp["AUC_local_40k"], w, label="40,000 (local)",
       color="#0072B2", edgecolor="black", linewidth=0.6)
ax.bar(x + w/2, comp["AUC_full_243k"], w, label="243,323 (full)",
       color="#D55E00", edgecolor="black", linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels([c.split("_")[0] for c in comp["GHS_Column"]], rotation=45)
ax.set_ylabel("AUC-ROC"); ax.set_ylim(0.5, 1.02)
ax.set_title("Effect of training-set size on GHS hazard prediction",
             fontweight="bold")
ax.legend(); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "colab_size_comparison.png"), dpi=300,
            bbox_inches="tight")
plt.show()

---
## 11. SHAP interpretability on the full-data model

Confirms whether the structure-hazard relationships found locally survive
training on the full dataset. The strongest local finding was **MACCS key 124**
(`[!#6;!#1]~[!#6;!#1]`, two directly bonded heteroatoms) as the top predictor
for both explosives and oxidisers — the signature of nitro, nitrate, peroxide
and azide groups.

In [ ]:
import shap

SAMPLE = 500
rng = np.random.RandomState(RANDOM_SEED)
pick = np.sort(rng.choice(len(test_idx), min(SAMPLE, len(test_idx)), replace=False))
X_explain = X[test_idx][pick]

shap_rows = []
for i, col in enumerate(GHS_LABEL_COLUMNS):
    if models[i] is None:
        continue
    vals = shap.TreeExplainer(models[i]).shap_values(X_explain,
                                                     check_additivity=False)
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[:, :, 1]
    mean_abs = np.abs(vals).mean(axis=0)
    for rank, j in enumerate(np.argsort(mean_abs)[::-1][:10], 1):
        fv, sv = X_explain[:, j], vals[:, j]
        corr = (np.corrcoef(fv, sv)[0, 1]
                if fv.std() > 1e-12 and sv.std() > 1e-12 else 0.0)
        shap_rows.append({
            "GHS_Column": col, "Rank": rank, "Feature": kept_names[j],
            "Mean_Abs_SHAP": round(float(mean_abs[j]), 6),
            # Direction comes from the correlation between the descriptor value
            # and its SHAP value - NOT from the mean signed SHAP, which is
            # negative for almost every feature on a rare class regardless of
            # the chemistry.
            "Value_SHAP_Correlation": round(float(corr), 4),
            "Direction": ("higher -> more hazardous" if corr > 0.05
                          else "higher -> less hazardous" if corr < -0.05
                          else "non-monotonic"),
        })

shap_df = pd.DataFrame(shap_rows)
shap_df.to_csv(os.path.join(OUT_DIR, "colab_shap_top10.csv"), index=False)
print(shap_df[shap_df.Rank <= 3].to_string(index=False))

---
## 12. What you now have in Drive

Everything is under `MyDrive/GHS_Project/colab_full_run/`:

| File | What it is |
|---|---|
| `colab_full_results.csv` | Full-dataset metrics, all nine classes |
| `colab_vs_local_comparison.csv` | 243k vs 40k, with significance flags |
| `colab_size_comparison.png` | The comparison figure |
| `colab_shap_top10.csv` | Top ten SHAP features per class |
| `colab_xgb_full.pkl` | The nine trained models |
| `colab_X_full.npy` | Cached descriptors (skip section 5 next time) |

### How to write this up

If section 10 reported **0 of 9 classes changed meaningfully**, the memory
constraint is no longer a limitation — you have shown the subset was
sufficient. Say so directly in the Methods, and cite the comparison table.

If some classes did improve, lead with the full-dataset numbers and keep the
40,000 result as a dataset-size ablation. Either outcome is publishable; the
second is a stronger paper.

---

**Disclaimer.** This prediction framework is a computational screening tool and
does not replace laboratory testing or regulatory assessment under Malaysia's
CLASS Regulations 2013.
